# Integración de una ontología como una base de conocimiento en un MAS

## Objetivo general

Que el(la) estudiante implemente la integración de una ontología como base de conocimiento en un ecosiste a multiagente que socialice todos sus componentes (clases, sub-clase, propiedades de los objetos, propiedades de los datos, axiomas), a través de la acción de un agente especializado que consulte y valide información. 

In [1]:
print("Hola mundo")

Hola mundo


### Importación de dependencias

In [1]:
import os
from crewai import Agent, Crew, Process, Task, LLM
from crewai.tools import tool
import owlready2 as owl

In [3]:
llm = LLM(
    model = "ollama/llama3.2",
    base_url = "http://localhost:11434",
)
print("Modelo funcionando correctamente")

Modelo funcionando correctamente


### Definición de las herramientas (tools)

In [2]:
@tool("Consultar la ontología académica de la Universidad Anáhuac México")
def consultar_ontologia(tipo_consulta: str, id_entidad: str = "") -> str:
    """
    Consulta la ontología académica de la Universidad Anáhuac México.
    
    Parámetros:
        - tipo_consulta: 'buscar_alumno', 'materias_inscritas', 'profesor_materia', 'resumen_clases', etc.
        - id_entidad: El identificador de la entidad a consultar o individuo. por ejemplo, alumno_001, profesor_001, materia_001.
    """

    # Cargar la ontología
    ruta_ontologia = "ontologia-ia-anahuac-ejemplo-1.owx"
    if not os.path.exists(ruta_ontologia):
        return f"Error: No se encontró la ontología en la ruta {ruta_ontologia}."

    try:
        ontologia = owl.get_ontology(ruta_ontologia).load()
        # Definición de las consultas según el tipo de consulta
        if tipo_consulta == "buscar_alumno":
            alumno = ontologia.search_one(iri=f"*{id_entidad}")
            if alumno:
                nombre = alumno.tieneNombre if hasattr(alumno, 'tieneNombre') else "Nombre no disponible"
                cuenta = alumno.tieneNoCuenta if hasattr(alumno, 'tieneNoCuenta') else "Cuenta no disponible"
                return f"Alumno encontrado: {nombre}, No. de Cuenta: {cuenta}"
            return "Alumno no encontrado."
        elif tipo_consulta == "materias_inscritas":
            alumno = ontologia.search_one(iri=f"*{id_entidad}")
            if alumno and hasattr(alumno, 'estaInscritoEn'):
                materias = [materia.tieneNombreMateria for materia in alumno.estaInscritoEn]
                return f"Materias inscritas: {', '.join(materias)}"
            return "Alumno no encontrado o no tiene materias inscritas."
        elif tipo_consulta == "profesor_materia":
            materia = ontologia.search_one(iri=f"*{id_entidad}")
            if materia and hasattr(materia, 'impartidaPor'):
                profesores = [profesor.tieneNombre for profesor in materia.impartidaPor]
                nombre_profesor = profesores[0] if profesores else "Nombre no disponible"
                return f"Profesor de la materia: {nombre_profesor}"
            return "Materia no encontrada o no tiene profesor asignado."
        elif tipo_consulta == "resumen_clases":
            clases = [clase.name for clase in ontologia.classes()]
            return f"Resumen de clases: {', '.join(clases)}"
        else:
            return "Tipo de consulta no reconocida."
    except Exception as e:
        return f"Error al cargar la ontología: {str(e)}"
    

### Definición de agentes

En este escenario se definirá 3 agentes especializados, los cuales realizarán las siguientes tareas:
- Agente Consultor (analiza la semántica de la ontología)
- Agente Validador (analiza las reglas o axiomas definidos en la ontología)
- Agente Coordinador (Gestiona las consultas para dar respuestas por ejemplo a usuarios del Depto. de control escolar)

In [6]:
# Agente consultor semántico
agente_consultor = Agent(
    role = "Especialista en Razonamiento Semántico y Ontologías",
    goal = "Interrogar la ontología académica de la Universidad Anáhuac México para obtener información sobre alumnos, materias y profesores.",
    backstory = ("Eres un agente experto en ontologías y razonamiento semántico, capaz de realizar consultas complejas sobre la estructura académica de la Universidad Anáhuac México. En caso de no tener respuestas a las preguntas del usuario, debes indicar que no tienes la información disponible y no inventar datos. Debes ser preciso y claro en tus respuestas, proporcionando información basada en la ontología disponible."),
    tools = [consultar_ontologia],
    llm = llm,
    verbose = True
)

In [7]:
# Agente validador de consultas
agente_validador = Agent(
    role = "Eres un Auditor de Consultas Ontológicas",
    goal = "Validar la precisión y relevancia de las consultas realizadas por el agente consultor semántico, asegurando que las respuestas sean coherentes con la ontología académica de la Universidad Anáhuac México. Cumpliendo con las restricciones dadas por la logica proposicional y la estructura de la ontología.",
    backstory = ("Eres un auditor académico de la Universidad Anáhuac México, especializado en ontologías y razonamiento semántico. Tu tarea es revisar las consultas realizadas por el agente consultor semántico y asegurarte de que las respuestas sean precisas, relevantes y coherentes con la ontología académica. Debes proporcionar retroalimentación constructiva y sugerencias para mejorar la calidad de las consultas."),
    tools = [consultar_ontologia],
    llm = llm,
    verbose = True
)